<a href="https://colab.research.google.com/github/sarthak-geek/Book_recommendation_system/blob/main/Notebooks/Popularity_based_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import kagglehub
import os
import matplotlib.pyplot as plt
import pickle
from google.colab import files

In [3]:
def get_dataset():
#Download Dataset from kaggle
  path = kagglehub.dataset_download("arashnic/book-recommendation-dataset")
  #import dataset and assign to repective variable
  books = pd.read_csv(os.path.join(path,'Books.csv'))
  users = pd.read_csv(os.path.join(path,'Users.csv'))
  ratings = pd.read_csv(os.path.join(path,'Ratings.csv'))
  return (books, users, ratings)


In [4]:
(books, users, ratings) = get_dataset()

100%|██████████| 24.3M/24.3M [00:01<00:00, 21.4MB/s]

Extracting files...



/tmp/ipykernel_10008/1297533687.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv(os.path.join(path,'Books.csv'))


In [5]:
#merging ratings and book dataset on top of ISBN column
ratings_book_merge = ratings.merge(books, on='ISBN')
print(ratings_book_merge.shape)
ratings_book_merge.head()

(1031136, 10)


,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...
2,276727,0446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...


In [6]:
#Checking how many users have rated each book
rating_count = ratings_book_merge.groupby("Book-Title").count()['User-ID'].reset_index() #reset_index here helps to converts the series back into df
#Checking average rating for each book
mean_rating = ratings_book_merge.groupby("Book-Title")['Book-Rating'].mean().reset_index()
#merging the two datasets
book_avg_ratings = rating_count.merge(mean_rating, on="Book-Title")
#only considering books rated by atleast 250 users
popular_books = book_avg_ratings[book_avg_ratings['User-ID'] >= 250]
#acquiring top 50 books with their title, author, total ratings and average rating and image
popular_books = popular_books.sort_values('Book-Rating', ascending=False).head(50)
popular_books = popular_books.merge(books, on="Book-Title").drop_duplicates('Book-Title')
popular_books = popular_books.rename(columns={'User-ID':'num_ratings', "Book-Rating":"avg_rating"})
popular_books = popular_books[['Book-Title','Book-Author','Image-URL-M', 'num_ratings', 'avg_rating']]

In [ ]:
popular_books

,Book-Title,Book-Author,Image-URL-M,num_ratings,avg_rating
0,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,http://images.amazon.com/images/P/0439136350.0...,428,5.852804
3,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,http://images.amazon.com/images/P/0439139597.0...,387,5.824289
5,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,http://images.amazon.com/images/P/0590353403.0...,278,5.737410
9,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,http://images.amazon.com/images/P/043935806X.0...,347,5.501441
13,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,http://images.amazon.com/images/P/0439064872.0...,556,5.183453
16,The Hobbit : The Enchanting Prelude to The Lor...,J.R.R. TOLKIEN,http://images.amazon.com/images/P/0345339681.0...,281,5.007117
17,The Fellowship of the Ring (The Lord of the Ri...,J.R.R. TOLKIEN,http://images.amazon.com/images/P/0345339703.0...,368,4.948370
26,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,http://images.amazon.com/images/P/059035342X.0...,575,4.895652
28,"The Two Towers (The Lord of the Rings, Part 2)",J.R.R. TOLKIEN,http://images.amazon.com/images/P/0345339711.0...,260,4.880769
39,To Kill a Mockingbird,Harper Lee,http://images.amazon.com/images/P/0446310786.0...,510,4.700000


In [8]:
with open('popular_books.pkl', 'wb') as f:
  pickle.dump(popular_books, f)
files.download("popular_books.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>